In [1]:
import os
# Disable symlink warnings on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Suppress all Hugging Face logs/warnings
from transformers.utils import logging
logging.set_verbosity_error()

In [2]:
# STEP 1: Imports
# ===============================
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [5]:
# Load clinical summarization model
print("Loading model...")

model_name = "google/flan-t5-base"  # public, free, CPU-friendly

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

print(" Model loaded")

Loading model...
 Model loaded


In [11]:
# Step 3: SOAP-style summarizer
# ===============================
def soap_summarizer(note):
    # Strong prompt for SOAP formatting
    prompt = (
        "Summarize the following patient note into a SOAP format. "
        "Output in clear sections: Subjective, Objective, Assessment, Plan. "
        "Be concise and clinically accurate:\n\n" + note
    )

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=120,
            min_length=50,
            num_beams=6,
            length_penalty=1.0,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
# Example note
note = """Patient is a 54-year-old male presenting with chest pain radiating to the 
left arm, shortness of breath for 2 hours. BP 150/95, HR 102. ECG shows ST elevation. 
Troponin elevated at 2.3. Impression: Acute STEMI. 
Plan: cath lab activation, aspirin 325mg given, heparin started."""

clinic_summary = soap_summarizer(note)

print("\n Clinical Summary:\n")
print(clinic_summary)


 Clinical Summary:

Acute chest pain radiating to the left arm, shortness of breath for 2 hours. BP 150/95, HR 102. ECG shows ST elevation. Troponin elevated at 2.3. Plan: cath lab activation, aspirin 325mg given, heparin started.
